In [26]:
import tensorflow as tf
import torch
import torch.nn.functional as F
import numpy as np

from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet import preprocess_input

In [27]:
keras_model_path = r"D:\Final Year\Sem 2\Major Project\Model Project\Model B (Keras Tensorflow)\keras_model_with_5k_images.h5"
pytorch_model_path = r"D:\Final Year\Sem 2\Major Project\Model Project\Model A (MobileNet)\mobilenet_model.pth"
student_model_path = r"D:\Final Year\Sem 2\Major Project\Model Project\baseStudentModel\student_model.h5"

train_path = r"D:\Final Year\Sem 2\Major Project\Model Project\Dataset\train\images"
val_path = r"D:\Final Year\Sem 2\Major Project\Model Project\Dataset\valid\images"

In [28]:
keras_teacher = load_model(keras_model_path)
keras_teacher.trainable = False

In [29]:
import torchvision.models as models
import torch
import torch.nn as nn

pytorch_teacher = models.mobilenet_v2(pretrained=False)

# ✅ FIX: Force correct classifier (2 classes)
pytorch_teacher.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(pytorch_teacher.last_channel, 2)
)

# ✅ Load weights
pytorch_teacher.load_state_dict(
    torch.load(pytorch_model_path, map_location=torch.device('cpu')),
    strict=False
)

pytorch_teacher.eval()

C:\temp\ipykernel_19340\3937047466.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(pytorch_model_path, map_location=torch.device('cpu')),


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [30]:
dummy = torch.randn(1, 3, 224, 224)
out = pytorch_teacher(dummy)
print(out.shape)

torch.Size([1, 2])


In [31]:
student_model = load_model(student_model_path)

In [32]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    val_path,
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

train_data = train_data.map(lambda x, y: (preprocess_input(x), y))
val_data = val_data.map(lambda x, y: (preprocess_input(x), y))

Found 10000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [33]:
optimizer = tf.keras.optimizers.Adam(1e-4)
loss_fn = tf.keras.losses.CategoricalCrossentropy()

EPOCHS = 5
alpha = 0.5   # balance between true labels & teachers

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    for images, labels in train_data:

        # -------- Teacher Predictions --------
        
        # Keras teacher
        keras_preds = keras_teacher(images, training=False)

        # PyTorch teacher
        torch_images = torch.tensor(images.numpy()).permute(0,3,1,2).float()
        with torch.no_grad():
            torch_preds = pytorch_teacher(torch_images)
            torch_preds = F.softmax(torch_preds, dim=1).numpy()

        # Combine teachers
        teacher_preds = (keras_preds.numpy() + torch_preds) / 2.0

        # -------- Student Training --------
        with tf.GradientTape() as tape:
            student_preds = student_model(images, training=True)

            loss_true = loss_fn(labels, student_preds)
            loss_teacher = loss_fn(teacher_preds, student_preds)

            loss = alpha * loss_true + (1 - alpha) * loss_teacher

        grads = tape.gradient(loss, student_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, student_model.trainable_variables))

    print("Epoch completed")


Epoch 1/5


Epoch completed

Epoch 2/5
Epoch completed

Epoch 3/5
Epoch completed

Epoch 4/5
Epoch completed

Epoch 5/5
Epoch completed


In [34]:
loss, acc = student_model.evaluate(val_data)
print("Final Accuracy:", acc)

63/63 [==============================] - 38s 580ms/step - loss: 0.1628 - accuracy: 0.9935
Final Accuracy: 0.9934999942779541


In [35]:
student_model.save("hybrid_model.h5")

c:\Users\Hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [36]:
import os
print(os.path.exists("hybrid_model.h5"))

True


In [37]:
from tensorflow.keras.models import load_model

model = load_model("hybrid_model.h5")
print("Model loaded successfully")

Model loaded successfully


In [38]:
loss, acc = model.evaluate(val_data)
print("Hybrid Model Accuracy:", acc)

63/63 [==============================] - 44s 669ms/step - loss: 0.1628 - accuracy: 0.9935
Hybrid Model Accuracy: 0.9934999942779541


In [40]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet import preprocess_input

img_path = r"D:\Final Year\Sem 2\Major Project\Model Project\Dataset\valid\images\Face\man.jpg"

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

pred = model.predict(img_array)

print("Prediction:", pred)
print("Class:", "Face" if np.argmax(pred) == 0 else "Non-Face")

1/1 [==============================] - 1s 535ms/step
Prediction: [[0.84836787 0.15163215]]
Class: Face


In [42]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet import preprocess_input

img_path = r"D:\Final Year\Sem 2\Major Project\Model Project\Dataset\valid\images\Non-Face\iphone.jpg"

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

pred = model.predict(img_array)

print("Prediction:", pred)
print("Class:", "Face" if np.argmax(pred) == 0 else "Non-Face")

1/1 [==============================] - 0s 55ms/step
Prediction: [[0.11858107 0.8814189 ]]
Class: Non-Face


In [46]:
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    label_mode='categorical',
    image_size=(224, 224),
    batch_size=32
)

# ✅ SAVE CLASS NAMES HERE
class_names = train_data.class_names

# THEN apply preprocessing
train_data = train_data.map(lambda x, y: (preprocess_input(x), y))
print(class_names)

Found 10000 files belonging to 2 classes.
['Face', 'Non-Face']
